# Project 2 OpenCV Experiments

Upload one photo of your tape track to Colab and name it `test_frame.jpg`. Use a photo taken from roughly the same forward-down camera angle you will use on the robot.

This notebook lets you calibrate the lane-detection pipeline on a still image before running the robot.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow

# Load your photo.
frame = cv2.imread("test_frame.jpg")
if frame is None:
    raise FileNotFoundError("Upload your track photo and name it test_frame.jpg")

print(f"Image size: {frame.shape[1]} x {frame.shape[0]} pixels")

# OpenCV uses BGR order. Matplotlib expects RGB.
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(10, 6))
plt.imshow(frame_rgb)
plt.title("Your camera frame")
plt.axis("off")
plt.show()

## Cell 2: Convert to grayscale

The lane detector only needs brightness. Grayscale turns each pixel into one value from 0 to 255.

In [ ]:
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(frame_rgb)
plt.title("Original color")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(gray, cmap="gray")
plt.title("Grayscale")
plt.axis("off")
plt.tight_layout()
plt.show()

print(f"Color image shape: {frame.shape}")
print(f"Grayscale shape:   {gray.shape}")

## Cell 3: Gaussian blur

Blur removes tiny sensor noise before thresholding. Try `(3, 3)` for mild blur or `(15, 15)` for heavy blur.

In [ ]:
blur = cv2.GaussianBlur(gray, (7, 7), 0)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(gray, cmap="gray")
plt.title("Before blur")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(blur, cmap="gray")
plt.title("After 7x7 Gaussian blur")
plt.axis("off")
plt.tight_layout()
plt.show()

## Cell 4: Binary threshold

This is the key calibration cell. Change `BINARY_THRESHOLD` and run the cell again until only the tape is white and the floor is black.

Write down the value that works. That value goes into `lane_follower_adv.py`.

In [ ]:
BINARY_THRESHOLD = 70  # Change this value and run again.

_, binary = cv2.threshold(blur, BINARY_THRESHOLD, 255, cv2.THRESH_BINARY_INV)

plt.figure(figsize=(14, 5))
plt.subplot(1, 3, 1)
plt.imshow(gray, cmap="gray")
plt.title("Grayscale")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(blur, cmap="gray")
plt.title("Blurred")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(binary, cmap="gray")
plt.title(f"Binary threshold={BINARY_THRESHOLD}\nwhite = tape, black = floor")
plt.axis("off")
plt.tight_layout()
plt.show()

white_pixels = np.count_nonzero(binary)
total_pixels = binary.size
print(f"White pixels: {white_pixels} ({100 * white_pixels / total_pixels:.1f}% of image)")
print("These are the pixels the algorithm thinks are tape")

Too low, such as `40`: too few pixels become white and parts of the tape may disappear.

Too high, such as `120`: too many pixels become white and the floor may get counted as tape.

`cv2.THRESH_BINARY_INV` is used because dark tape should become white in the binary image. For white tape on a dark floor, use `cv2.THRESH_BINARY` in the robot code instead.

## Cell 5: Morphological cleaning

`OPEN` removes small white specks. `CLOSE` fills small gaps in the white tape region.

In [ ]:
kernel = np.ones((7, 7), np.uint8)
opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
clean = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, title in zip(
    axes,
    [binary, opened, clean],
    ["After threshold", "After OPEN: noise removed", "After CLOSE: gaps filled"],
):
    ax.imshow(img, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

Try changing the kernel to `(3, 3)`, `(9, 9)`, or `(11, 11)`. A larger kernel removes more noise but can damage a thin tape line.

## Cell 6: Row scanning

This finds the center of the tape on several horizontal rows. The magenta dots should sit on the tape.

In [ ]:
h, w = clean.shape
ROW_FRACS = [0.90, 0.75, 0.60, 0.45, 0.30]

display = cv2.cvtColor(clean, cv2.COLOR_GRAY2BGR)
points = []

for frac in ROW_FRACS:
    ry = int(h * frac)
    xs = np.where(clean[ry] > 0)[0]

    cv2.line(display, (0, ry), (w, ry), (255, 150, 0), 1)

    if len(xs) < 15:
        print(f"Row {frac:.0%}: only {len(xs)} white pixels, skipped")
        continue

    runs = np.split(xs, np.where(np.diff(xs) > 1)[0] + 1)
    best = max(runs, key=len)

    if len(best) < 12:
        print(f"Row {frac:.0%}: longest run only {len(best)} pixels, skipped")
        continue

    cx = int((best[0] + best[-1]) / 2)
    points.append((cx, ry))

    cv2.circle(display, (cx, ry), 8, (255, 0, 255), -1)
    print(f"Row {frac:.0%}: tape center at x={cx} (run width: {len(best)} px)")

if len(points) >= 2:
    cv2.line(display, points[0], points[-1], (0, 100, 255), 2)
    print(f"\nNear point: {points[0]}")
    print(f"Far point:  {points[-1]}")
    print(f"Horizontal shift: {points[-1][0] - points[0][0]} px")

plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(display, cv2.COLOR_BGR2RGB))
plt.title("Row scanning: blue=scan lines, magenta=centers, orange=direction")
plt.axis("off")
plt.show()

If the dots are wrong, go back to Cell 4 and tune `BINARY_THRESHOLD`. If no dots appear even though the tape is visible, reduce the 15-pixel and 12-pixel minimums.

## Cell 7: Two-point steering error

The robot computes this same number every frame while it is moving.

In [ ]:
if len(points) >= 2:
    near_cx = points[0][0]
    far_cx = points[-1][0]

    position_error = (near_cx - w / 2) / w
    direction_error = (far_cx - near_cx) / w
    raw_angle_error = 0.25 * position_error + 0.75 * direction_error

    print("=" * 50)
    print(f"Frame center:    x = {w // 2}")
    print(f"Near point:      x = {near_cx}")
    print(f"Far point:       x = {far_cx}")
    print()
    print(f"Position error:  {position_error:+.3f}")
    print(f"Direction error: {direction_error:+.3f}")
    print(f"Combined error:  {raw_angle_error:+.3f}")
    print()
    if raw_angle_error > 0.05:
        print("Robot should steer RIGHT")
    elif raw_angle_error < -0.05:
        print("Robot should steer LEFT")
    else:
        print("Line is approximately centered, robot drives straight")
else:
    print("Not enough points detected to compute steering error")
    print("Check your BINARY_THRESHOLD value")

## Values to take back to the robot code

- `BINARY_THRESHOLD = _____`
- Kernel size working well: `_____ x _____`
- Typical points count on a straight line: `_____`

Put these values into the configuration section of `lane_follower_adv.py` before starting the robot.